# 🌍 Zero Hunger — Big Data Analytics Pipeline
## Complete End-to-End Notebook: All 6 Pipeline Stages

**Course:** M.S. Data Science — Big Data Analytics  
**Institution:** University of Houston–Clear Lake  
**Authors:** Govardhan Reddy Narala · Divya Keelanur · Ramya Vankayalapati · Rojesh Thapa · Tharun Athota

---

### Pipeline Architecture

```
Raw Data Sources (FAO / World Bank / UNICEF)
             ↓
  Stage 1 – Data Cleaning & Normalization
             ↓
    HDFS Storage (Distributed)
             ↓
Hadoop Streaming MapReduce (Python)
             ↓
  Stage 2 – Hive Queries (8 SDG indicators)
             ↓
  Stage 3 – Spark Aggregations
             ↓
  Stage 4 – Machine Learning Forecasting
             ↓
  Stage 5 – Visualization & Dashboard
             ↓
  Stage 6 – Policy Insights & Recommendations
```

### Notebook Overview

| Stage | Description | Key Outputs |
|-------|-------------|-------------|
| Stage 1 | Data Cleaning & Normalization | `data/processed/features.csv` |
| Stage 2 | Hive Queries | 8 SDG query result tables |
| Stage 3 | Spark Aggregations | Regional/yearly aggregates |
| Stage 4 | Machine Learning | LR & RF models, metrics |
| Stage 5 | Visualization | Charts (PNG) + Dashboard (HTML) |
| Stage 6 | Policy Insights | Report (MD) + Insights (JSON) |


## 📦 0. Install Dependencies

In [ ]:
# Install all required packages (Google Colab / fresh environment)
import subprocess, sys

packages = [
    "pandas>=1.5.0",
    "numpy>=1.23.0",
    "matplotlib>=3.6.0",
    "seaborn>=0.12.0",
    "plotly>=5.11.0",
    "scikit-learn>=1.1.0",
    "kaleido",          # static image export for plotly
    "tabulate",         # formatted table printing
    "openpyxl",         # Excel file support
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# PySpark (skip if already installed)
try:
    import pyspark
    print(f"PySpark {pyspark.__version__} already installed")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.3"])

print("\n✅ All dependencies installed successfully.")


## 📚 1. Import All Required Libraries

In [ ]:
import os
import sys
import json
import warnings
import subprocess
from pathlib import Path
from datetime import datetime, date
from collections import defaultdict

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.linear_model import LinearRegression as SKLinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer

# PySpark
try:
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    from pyspark.sql.types import (
        StructType, StructField, StringType, IntegerType, DoubleType
    )
    SPARK_AVAILABLE = True
    print("✅ PySpark imported successfully")
except ImportError:
    SPARK_AVAILABLE = False
    print("⚠️  PySpark not available – Spark stages will use pandas fallback")

warnings.filterwarnings("ignore")
matplotlib.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13})
sns.set_theme(style="whitegrid", palette="husl")

print(f"pandas {pd.__version__} | numpy {np.__version__} | "
      f"matplotlib {matplotlib.__version__} | seaborn {sns.__version__}")


## ⚙️ 2. Configuration & Data Paths

Edit the paths below to match your environment.  
**Google Colab**: mount Google Drive and set `DRIVE_ROOT` to your data folder.  
**Local / Jupyter**: set `RAW_DATA_DIR` to the folder containing the CSV files.


In [ ]:
# ─── Google Colab: mount Drive (skip if running locally) ───────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = '/content/drive/MyDrive/hunger_data'
    print("Google Drive mounted at /content/drive")
except Exception:
    DRIVE_ROOT = None
    print("Not running in Colab – using local paths")

# ─── Data paths ─────────────────────────────────────────────────────────────
RAW_DATA_DIR       = os.environ.get("RAW_DATA_DIR",       DRIVE_ROOT or "data/raw")
PROCESSED_DATA_DIR = os.environ.get("PROCESSED_DATA_DIR", "data/processed")
OUTPUT_SPARK_DIR   = os.environ.get("OUTPUT_SPARK_DIR",   "output/spark")
OUTPUT_ML_DIR      = os.environ.get("OUTPUT_ML_DIR",      "output/ml")
OUTPUT_VIZ_DIR     = os.environ.get("OUTPUT_VIZ_DIR",     "output/visualizations")
OUTPUT_POLICY_DIR  = os.environ.get("OUTPUT_POLICY_DIR",  "output/policy_insights")

SDG_CSV    = os.path.join(RAW_DATA_DIR, "SDG_BulkDownloads_E_All_Data_Normalized_cleaned.csv")
FAOSTAT_CSV = os.path.join(RAW_DATA_DIR, "FAOSTAT_data_en_11-15-2024.csv")
HUNGER_CSV  = os.path.join(RAW_DATA_DIR, "hunger_index_interpolated.csv")

# HDFS paths (used when Hadoop is running)
HDFS_MR_OUTPUT   = os.environ.get("HDFS_MR_OUTPUT",   "/output")
HDFS_NAMENODE    = os.environ.get("HDFS_NAMENODE",    "hdfs://localhost:9000")

# Create output directories
for d in [PROCESSED_DATA_DIR, OUTPUT_SPARK_DIR, OUTPUT_ML_DIR, OUTPUT_VIZ_DIR, OUTPUT_POLICY_DIR]:
    os.makedirs(d, exist_ok=True)

print("Configuration:")
print(f"  RAW_DATA_DIR       = {RAW_DATA_DIR}")
print(f"  PROCESSED_DATA_DIR = {PROCESSED_DATA_DIR}")
print(f"  OUTPUT_SPARK_DIR   = {OUTPUT_SPARK_DIR}")
print(f"  OUTPUT_ML_DIR      = {OUTPUT_ML_DIR}")
print(f"  OUTPUT_VIZ_DIR     = {OUTPUT_VIZ_DIR}")
print(f"  OUTPUT_POLICY_DIR  = {OUTPUT_POLICY_DIR}")
print(f"  SDG CSV exists     = {os.path.exists(SDG_CSV)}")
print(f"  FAOSTAT CSV exists = {os.path.exists(FAOSTAT_CSV)}")
print(f"  Hunger CSV exists  = {os.path.exists(HUNGER_CSV)}")


---
## 🧹 Stage 1 — Data Cleaning & Normalization

This stage:
1. **Loads** raw CSV files from HDFS / local storage (FAO, SDG, Hunger Index)
2. **Handles missing values** with time-series interpolation and median imputation
3. **Detects and caps outliers** using the IQR (Tukey fence) method
4. **Normalizes numeric features** to [0, 1] with Min-Max scaling
5. **Removes duplicates**
6. **Generates a cleaning report** summarising all transformations

### SDG Item Codes used in this project

| Code | Indicator |
|------|-----------|
| 24001 | Prevalence of undernourishment |
| 24003 | Prevalence of moderate/severe food insecurity |
| 24004 | Number affected by severe food insecurity |
| 24005 | Number moderately/severely food insecure |
| 7004 | Cost of healthy diet (USD/day) |
| 7005 | Prevalence of food unaffordability (%) |
| 7006 | Number unable to afford healthy diet (millions) |
| 7007 | Cost of starchy staples (USD/day) |


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 1A – Load raw data from HDFS / local files
# ────────────────────────────────────────────────────────────────────────────

SDG_ITEM_CODES = {
    24001: "Prevalence_Undernourishment",
    24003: "Food_Insecurity_Pct",
    24004: "Severe_Food_Insecurity_N",
    24005: "Moderate_Severe_Food_Insecurity_N",
    7004:  "Cost_Healthy_Diet",
    7005:  "Food_Unaffordability_Pct",
    7006:  "Unable_Afford_Healthy_Diet_N",
    7007:  "Cost_Starchy_Staples",
}

RENAME_MAP = {
    "Area Code": "AreaCode", "Area Code (M49)": "AreaCode",
    "Item Code": "ItemCode", "Item Code (CPC)": "ItemCode",
    "Area": "Area", "Item": "Item", "Year": "Year",
    "Unit": "Unit", "Value": "Value", "Flag": "Flag",
}


def _try_hdfs_then_local(hdfs_path: str, local_path: str) -> str:
    """Return HDFS path if accessible, otherwise fall back to local path."""
    try:
        result = subprocess.run(
            ["hdfs", "dfs", "-test", "-e", hdfs_path],
            capture_output=True, timeout=10
        )
        if result.returncode == 0:
            print(f"  ✅ HDFS: {hdfs_path}")
            return f"hdfs:{hdfs_path}"
    except Exception:
        pass
    if os.path.exists(local_path):
        print(f"  ✅ Local: {local_path}")
        return local_path
    print(f"  ⚠️  Not found – {local_path} (will generate synthetic data)")
    return None


def load_sdg_bulk(filepath: str) -> pd.DataFrame:
    """Load and standardise the FAO SDG bulk download CSV."""
    df = pd.read_csv(filepath, low_memory=False)
    df.columns = df.columns.str.strip()
    df.rename(columns={k: v for k, v in RENAME_MAP.items() if k in df.columns}, inplace=True)
    for col in ("AreaCode", "ItemCode", "Year"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if "Value" in df.columns:
        df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
    return df


def load_faostat(filepath: str) -> pd.DataFrame:
    """Load FAOSTAT crop/production data."""
    df = pd.read_csv(filepath, low_memory=False)
    df.columns = df.columns.str.strip()
    if "Value" in df.columns:
        df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
    return df


def load_hunger_index(filepath: str) -> pd.DataFrame:
    """Load the interpolated Global Hunger Index dataset."""
    if filepath.endswith(".xlsx"):
        df = pd.read_excel(filepath)
    else:
        df = pd.read_csv(filepath, low_memory=False)
    df.columns = df.columns.str.strip()
    return df


def _make_synthetic_sdg(n_countries=50, years=range(2000, 2023)) -> pd.DataFrame:
    """Generate reproducible synthetic data for demo/testing."""
    rng = np.random.default_rng(42)
    rows = []
    areas = [f"Country_{i:02d}" for i in range(n_countries)]
    for area_code, area in enumerate(areas, 1):
        for year in years:
            for item_code, item in SDG_ITEM_CODES.items():
                base_val = rng.uniform(3, 45)
                trend = rng.uniform(-0.3, 0.1) * (year - 2000)
                val = max(0, base_val + trend + rng.normal(0, 1))
                rows.append({"AreaCode": area_code, "Area": area,
                             "ItemCode": item_code, "Item": item,
                             "Year": year, "Value": val, "Unit": "%"})
    df = pd.DataFrame(rows)
    # inject some missing values (~3%)
    mask = rng.random(len(df)) < 0.03
    df.loc[mask, "Value"] = np.nan
    return df


print("Loading raw data...")
if os.path.exists(SDG_CSV):
    sdg_raw = load_sdg_bulk(SDG_CSV)
else:
    print("  Using synthetic SDG data (no file found)")
    sdg_raw = _make_synthetic_sdg()

if os.path.exists(FAOSTAT_CSV):
    faostat_raw = load_faostat(FAOSTAT_CSV)
else:
    print("  Using synthetic FAOSTAT data")
    rng = np.random.default_rng(99)
    faostat_raw = pd.DataFrame({
        "Area":  np.repeat([f"Country_{i:02d}" for i in range(50)], 23),
        "Year":  list(range(2000, 2023)) * 50,
        "Value": rng.uniform(1000, 8000, 50 * 23),
    })

if os.path.exists(HUNGER_CSV):
    hunger_raw = load_hunger_index(HUNGER_CSV)
else:
    print("  Using synthetic Hunger Index data")
    rng = np.random.default_rng(77)
    hunger_raw = pd.DataFrame({
        "Area":        np.repeat([f"Country_{i:02d}" for i in range(50)], 23),
        "Year":        list(range(2000, 2023)) * 50,
        "Hunger_Index": rng.uniform(5, 55, 50 * 23),
    })

print(f"\nData loaded:")
print(f"  SDG rows:     {len(sdg_raw):,}  | cols: {list(sdg_raw.columns)[:6]}")
print(f"  FAOSTAT rows: {len(faostat_raw):,}  | cols: {list(faostat_raw.columns)[:6]}")
print(f"  Hunger rows:  {len(hunger_raw):,}  | cols: {list(hunger_raw.columns)[:6]}")


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 1B – Filter, Missing-value Handling, Interpolation
# ────────────────────────────────────────────────────────────────────────────

def report_missing(df: pd.DataFrame, label: str = "") -> pd.DataFrame:
    """Return a summary DataFrame of missing values per column."""
    missing = df.isnull().sum()
    pct = (missing / len(df) * 100).round(2)
    report = pd.DataFrame({"Column": missing.index,
                            "Missing_Count": missing.values,
                            "Missing_Pct": pct.values})
    report = report[report["Missing_Count"] > 0].reset_index(drop=True)
    if label:
        print(f"  [{label}] {len(report)} columns with missing values")
    return report


def filter_sdg_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only rows matching SDG food-security item codes."""
    if "ItemCode" not in df.columns:
        return df
    filtered = df[df["ItemCode"].isin(SDG_ITEM_CODES.keys())].copy()
    print(f"  filter_sdg_indicators: {len(df):,} → {len(filtered):,} rows")
    return filtered


def fill_time_series_gaps(df: pd.DataFrame,
                           group_cols: list,
                           value_col: str = "Value") -> pd.DataFrame:
    """Linear interpolation within groups, then ffill/bfill for edges."""
    df = df.sort_values(group_cols + ["Year"]).copy()
    df[value_col] = (
        df.groupby(group_cols)[value_col]
          .transform(lambda s: s.interpolate(method="linear").ffill().bfill())
    )
    return df


# ── Filter SDG ──
sdg_filtered = filter_sdg_indicators(sdg_raw)

# ── Missing value report (before) ──
print("\nMissing values BEFORE cleaning:")
mv_sdg     = report_missing(sdg_filtered, "SDG")
mv_faostat = report_missing(faostat_raw, "FAOSTAT")
mv_hunger  = report_missing(hunger_raw, "Hunger")
for label, mv in [("SDG", mv_sdg), ("FAOSTAT", mv_faostat), ("Hunger", mv_hunger)]:
    if mv.empty:
        print(f"  {label}: ✅ no missing values")
    else:
        print(f"  {label}:\n{mv.to_string(index=False)}")

# ── Interpolation / imputation ──
def impute_numeric(df, strategy="median"):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        imp = SimpleImputer(strategy=strategy)
        df = df.copy()
        df[numeric_cols] = imp.fit_transform(df[numeric_cols])
    return df

if "ItemCode" in sdg_filtered.columns and "AreaCode" in sdg_filtered.columns:
    sdg_clean = fill_time_series_gaps(sdg_filtered, ["AreaCode", "ItemCode"])
else:
    sdg_clean = sdg_filtered.copy()
sdg_clean = impute_numeric(sdg_clean)

faostat_clean = impute_numeric(faostat_raw)
hunger_clean  = impute_numeric(hunger_raw)

# ── Missing values AFTER ──
total_missing_after = (sdg_clean.isnull().sum().sum()
                       + faostat_clean.isnull().sum().sum()
                       + hunger_clean.isnull().sum().sum())
print(f"\nTotal missing values after imputation: {total_missing_after}")


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 1C – Outlier Detection & Capping, Deduplication, Normalization
# ────────────────────────────────────────────────────────────────────────────

def remove_outliers_iqr(df: pd.DataFrame, col: str, factor: float = 3.0) -> tuple:
    """Cap outliers with Tukey fences. Returns (cleaned_df, n_clipped)."""
    if col not in df.columns:
        return df, 0
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - factor * iqr, q3 + factor * iqr
    df = df.copy()
    n = int(((df[col] < lo) | (df[col] > hi)).sum())
    df[col] = df[col].clip(lo, hi)
    return df, n


def normalize_minmax(df: pd.DataFrame,
                     cols: list | None = None) -> tuple:
    """Min-Max scale to [0, 1]. Returns (scaled_df, scaler)."""
    if cols is None:
        cols = df.select_dtypes(include=[np.number]).columns.tolist()
    cols = [c for c in cols if c in df.columns]
    scaler = MinMaxScaler()
    df = df.copy()
    df[cols] = scaler.fit_transform(df[cols])
    return df, scaler


# ── Outlier handling ──
cleaning_report = {}

sdg_clean, n_sdg_outliers = remove_outliers_iqr(sdg_clean, "Value")
cleaning_report["sdg_outliers_capped"] = n_sdg_outliers
print(f"SDG outliers capped: {n_sdg_outliers}")

if "Value" in faostat_clean.columns:
    faostat_clean, n_fao_outliers = remove_outliers_iqr(faostat_clean, "Value")
    cleaning_report["faostat_outliers_capped"] = n_fao_outliers
    print(f"FAOSTAT outliers capped: {n_fao_outliers}")

if "Hunger_Index" in hunger_clean.columns:
    hunger_clean, n_hi_outliers = remove_outliers_iqr(hunger_clean, "Hunger_Index")
    cleaning_report["hunger_outliers_capped"] = n_hi_outliers
    print(f"Hunger Index outliers capped: {n_hi_outliers}")

# ── Deduplication ──
n_before = len(sdg_clean)
sdg_clean = sdg_clean.drop_duplicates()
cleaning_report["sdg_duplicates_removed"] = n_before - len(sdg_clean)
print(f"\nSDG duplicates removed: {n_before - len(sdg_clean)}")

# ── Build feature matrix (pivot SDG → wide format) ──
if "ItemCode" in sdg_clean.columns:
    pivot_cols = ["Area", "Year"] + (["AreaCode"] if "AreaCode" in sdg_clean.columns else [])
    sdg_pivot = (
        sdg_clean[sdg_clean["ItemCode"].isin(SDG_ITEM_CODES.keys())]
        .pivot_table(index=pivot_cols, columns="ItemCode", values="Value", aggfunc="mean")
        .reset_index()
    )
    sdg_pivot.columns.name = None
    # rename numeric item codes to descriptive names
    rename_cols = {code: label for code, label in SDG_ITEM_CODES.items()
                   if code in sdg_pivot.columns}
    sdg_pivot.rename(columns=rename_cols, inplace=True)
else:
    sdg_pivot = sdg_clean.copy()

# ── Merge with agricultural yield ──
ag_yield = (
    faostat_clean.groupby(["Area", "Year"])["Value"].mean()
    .reset_index().rename(columns={"Value": "Agricultural_Yield"})
) if "Area" in faostat_clean.columns and "Year" in faostat_clean.columns else pd.DataFrame()

if not ag_yield.empty:
    feature_df = sdg_pivot.merge(ag_yield, on=["Area", "Year"], how="left")
else:
    feature_df = sdg_pivot.copy()

# ── Merge hunger index ──
for alt in ("Country", "country", "Nation"):
    if alt in hunger_clean.columns and "Area" not in hunger_clean.columns:
        hunger_clean = hunger_clean.rename(columns={alt: "Area"})
for alt in ("GHI Score", "ghi", "GHI"):
    if alt in hunger_clean.columns and "Hunger_Index" not in hunger_clean.columns:
        hunger_clean = hunger_clean.rename(columns={alt: "Hunger_Index"})

if "Hunger_Index" in hunger_clean.columns and "Area" in hunger_clean.columns:
    hi_cols = ["Area", "Year", "Hunger_Index"]
    hi_cols = [c for c in hi_cols if c in hunger_clean.columns]
    feature_df = feature_df.merge(hunger_clean[hi_cols], on=["Area", "Year"], how="left")

# ── Final imputation on feature matrix ──
feature_df = impute_numeric(feature_df)

# ── Normalize ──
num_cols = feature_df.select_dtypes(include=[np.number]).columns.difference(
    ["AreaCode", "Year"]).tolist()
feature_df_norm, feat_scaler = normalize_minmax(feature_df, cols=num_cols)

# ── Save cleaned feature matrix ──
features_output_path = os.path.join(PROCESSED_DATA_DIR, "features.csv")
feature_df_norm.to_csv(features_output_path, index=False)

cleaning_report["final_shape"]           = feature_df_norm.shape
cleaning_report["normalized_columns"]    = len(num_cols)
cleaning_report["features_saved_to"]     = features_output_path

print(f"\n✅ Feature matrix shape: {feature_df_norm.shape}")
print(f"   Normalized {len(num_cols)} numeric columns")
print(f"   Saved → {features_output_path}")


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 1D – Cleaning Report
# ────────────────────────────────────────────────────────────────────────────

print("=" * 60)
print("  DATA CLEANING REPORT")
print("=" * 60)
print(f"  {'Metric':<40} {'Value':>15}")
print("-" * 60)

report_rows = {
    "SDG raw rows":               len(sdg_raw),
    "SDG filtered (SDG codes)":   len(sdg_filtered),
    "SDG duplicates removed":      cleaning_report.get("sdg_duplicates_removed", 0),
    "SDG outliers capped":         cleaning_report.get("sdg_outliers_capped", 0),
    "FAOSTAT outliers capped":     cleaning_report.get("faostat_outliers_capped", 0),
    "Hunger Index outliers capped":cleaning_report.get("hunger_outliers_capped", 0),
    "Feature matrix rows":         cleaning_report["final_shape"][0],
    "Feature matrix columns":      cleaning_report["final_shape"][1],
    "Normalized columns":          cleaning_report["normalized_columns"],
}

for metric, value in report_rows.items():
    print(f"  {metric:<40} {value:>15,}")

print("=" * 60)
print(f"  Output: {cleaning_report['features_saved_to']}")
print("=" * 60)

# Show sample
print("\nSample of cleaned feature matrix:")
display(feature_df_norm.head(5)) if 'display' in dir() else print(feature_df_norm.head(5).to_string())


---
## 🐝 Stage 2 — Hive Queries

This stage:
1. **Sets up** a SparkSession with Hive support (or uses pandas as fallback)
2. **Creates** Hive external tables pointing to MapReduce HDFS output
3. **Executes 8 comprehensive queries** covering all SDG food security indicators
4. **Displays results** in formatted tables

The full Hive SQL suite is in `hive/hive_queries.sql` (30+ queries, 7 SDG indicators).  
Here we execute the 8 most actionable queries using PySpark's Hive integration.


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 2A – SparkSession with Hive support
# ────────────────────────────────────────────────────────────────────────────

if SPARK_AVAILABLE:
    spark = (
        SparkSession.builder
        .appName("ZeroHungerHive")
        .config("spark.sql.warehouse.dir", "spark-warehouse")
        .config("spark.driver.memory", "2g")
        .config("spark.sql.adaptive.enabled", "true")
        .enableHiveSupport()
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("WARN")
    print(f"✅ SparkSession created  |  Spark {spark.version}")
    HIVE_MODE = True
else:
    print("⚠️  Spark not available – Hive queries will run as pandas SQL via pandasql")
    HIVE_MODE = False

# ────────────────────────────────────────────────────────────────────────────
# Stage 2B – Create Hive external tables from MapReduce output
# ────────────────────────────────────────────────────────────────────────────

HIVE_SCHEMA_DDL = """
CREATE DATABASE IF NOT EXISTS zero_hunger
"""

TABLES_DDL = {
    "sdg_hunger_indicators": {
        "ddl": """
            CREATE EXTERNAL TABLE IF NOT EXISTS zero_hunger.sdg_hunger_indicators (
                year          INT,
                area_code     INT,
                area          STRING,
                item_code     INT,
                item_code_sdg STRING,
                item          STRING,
                value         DOUBLE,
                unit          STRING,
                avg_value     DOUBLE,
                growth_rate   DOUBLE
            )
            ROW FORMAT DELIMITED
            FIELDS TERMINATED BY '\\t'
            STORED AS TEXTFILE
            LOCATION '/output'
        """,
        "description": "SDG Hunger Indicators (MapReduce Job 1 – /output)"
    },
    "cost_healthy_diet": {
        "ddl": """
            CREATE EXTERNAL TABLE IF NOT EXISTS zero_hunger.cost_healthy_diet (
                area     STRING,
                year     INT,
                cost_usd DOUBLE
            )
            ROW FORMAT DELIMITED
            FIELDS TERMINATED BY '\\t'
            STORED AS TEXTFILE
            LOCATION '/output1'
        """,
        "description": "Cost of Healthy Diet (Job 2 – /output1)"
    },
    "food_unaffordability": {
        "ddl": """
            CREATE EXTERNAL TABLE IF NOT EXISTS zero_hunger.food_unaffordability (
                area           STRING,
                year           INT,
                prevalence_pct DOUBLE
            )
            ROW FORMAT DELIMITED
            FIELDS TERMINATED BY '\\t'
            STORED AS TEXTFILE
            LOCATION '/output2'
        """,
        "description": "Food Unaffordability % (Job 3 – /output2)"
    },
}

if HIVE_MODE:
    try:
        spark.sql(HIVE_SCHEMA_DDL)
        print("✅ Database 'zero_hunger' created/verified")
        for tbl, info in TABLES_DDL.items():
            spark.sql(info["ddl"])
            print(f"   ✅ Table: {tbl}  – {info['description']}")
    except Exception as e:
        print(f"⚠️  Hive DDL failed (HDFS may not be running): {e}")
        HIVE_MODE = False
else:
    print("Hive table creation skipped (Spark not available)")


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 2C – Execute 8 Hive / SQL Queries on SDG data
# (Falls back to pandas if Spark/Hive is not available)
# ────────────────────────────────────────────────────────────────────────────

def run_hive_query(query_name: str, spark_query: str,
                   fallback_fn=None, show_n: int = 10):
    """
    Execute a Hive query via Spark, or call fallback_fn(feature_df) if Spark
    is not available.  Returns a pandas DataFrame.
    """
    print(f"\n{'─'*60}")
    print(f"  Query: {query_name}")
    print(f"{'─'*60}")
    if HIVE_MODE:
        try:
            result = spark.sql(spark_query).limit(show_n).toPandas()
            print(result.to_string(index=False))
            return result
        except Exception as e:
            print(f"  ⚠️  Spark query failed: {e}  →  using pandas fallback")
    if fallback_fn:
        result = fallback_fn(feature_df_norm)
        print(result.head(show_n).to_string(index=False))
        return result
    return pd.DataFrame()


# ── Helper to derive the "main" numeric column in feature_df ──
def _main_col():
    priority = ["Prevalence_Undernourishment", "Food_Insecurity_Pct",
                 "Hunger_Index", "Value"]
    for c in priority:
        if c in feature_df_norm.columns:
            return c
    numeric = feature_df_norm.select_dtypes(include=[np.number]).columns
    return numeric[0] if len(numeric) else None


MAIN_COL = _main_col()
AREA_COL = "Area" if "Area" in feature_df_norm.columns else feature_df_norm.columns[0]
YEAR_COL = "Year" if "Year" in feature_df_norm.columns else None


# ── Q1: Global yearly average ──
q1 = run_hive_query(
    "Q1 – Global yearly average undernourishment",
    """SELECT year, ROUND(AVG(value),3) AS global_avg
       FROM zero_hunger.sdg_hunger_indicators
       WHERE item_code = 24001
       GROUP BY year ORDER BY year""",
    fallback_fn=lambda df: (
        df.groupby(YEAR_COL)[MAIN_COL].mean().reset_index()
          .rename(columns={MAIN_COL: "global_avg"})
          .sort_values(YEAR_COL)
    ) if YEAR_COL else pd.DataFrame()
)

# ── Q2: Top-10 most food-insecure countries (latest year) ──
q2 = run_hive_query(
    "Q2 – Top-10 most food-insecure countries (latest year)",
    """SELECT area, ROUND(AVG(value),3) AS avg_prevalence
       FROM zero_hunger.sdg_hunger_indicators
       WHERE item_code = 24003
         AND year = (SELECT MAX(year) FROM zero_hunger.sdg_hunger_indicators)
       GROUP BY area
       ORDER BY avg_prevalence DESC LIMIT 10""",
    fallback_fn=lambda df: (
        df[df[YEAR_COL] == df[YEAR_COL].max()][[AREA_COL, MAIN_COL]]
          .sort_values(MAIN_COL, ascending=False)
          .head(10)
          .rename(columns={MAIN_COL: "avg_prevalence"})
    ) if YEAR_COL else pd.DataFrame()
)

# ── Q3: Worst 5-year growth rate by country ──
q3 = run_hive_query(
    "Q3 – Countries with worst 5-year growth rate",
    """SELECT area, ROUND(AVG(growth_rate),3) AS avg_growth
       FROM zero_hunger.sdg_hunger_indicators
       WHERE item_code = 24001 AND year >= 2018
       GROUP BY area
       ORDER BY avg_growth DESC LIMIT 10""",
    fallback_fn=lambda df: (
        df.groupby(AREA_COL)[MAIN_COL].mean().reset_index()
          .rename(columns={MAIN_COL: "avg_growth"})
          .sort_values("avg_growth", ascending=False).head(10)
    ) if AREA_COL else pd.DataFrame()
)

# ── Q4: Countries that improved most over the decade ──
q4 = run_hive_query(
    "Q4 – Countries that improved most (2013–2022)",
    """SELECT area,
              ROUND(MAX(CASE WHEN year=2013 THEN avg_value END),3) AS val_2013,
              ROUND(MAX(CASE WHEN year=2022 THEN avg_value END),3) AS val_2022,
              ROUND(MAX(CASE WHEN year=2013 THEN avg_value END)
                    - MAX(CASE WHEN year=2022 THEN avg_value END),3) AS improvement
       FROM zero_hunger.sdg_hunger_indicators
       WHERE item_code=24001 AND year IN (2013,2022)
       GROUP BY area
       HAVING improvement > 0
       ORDER BY improvement DESC LIMIT 10""",
    fallback_fn=lambda df: (
        df.groupby(AREA_COL).apply(
            lambda g: pd.Series({
                "improvement": g[g[YEAR_COL]==g[YEAR_COL].min()][MAIN_COL].mean()
                              - g[g[YEAR_COL]==g[YEAR_COL].max()][MAIN_COL].mean()
            })
        ).reset_index().sort_values("improvement", ascending=False).head(10)
    ) if YEAR_COL else pd.DataFrame()
)

# ── Q5: Regional summary (mean, max, min, count) ──
q5 = run_hive_query(
    "Q5 – Regional summary statistics",
    """SELECT area,
              COUNT(DISTINCT year) AS years_covered,
              ROUND(AVG(value),3)  AS mean_val,
              ROUND(MAX(value),3)  AS max_val,
              ROUND(MIN(value),3)  AS min_val
       FROM zero_hunger.sdg_hunger_indicators
       WHERE item_code=24001
       GROUP BY area
       ORDER BY mean_val DESC LIMIT 20""",
    fallback_fn=lambda df: (
        df.groupby(AREA_COL)[MAIN_COL].agg(
            years_covered="count", mean_val="mean",
            max_val="max", min_val="min"
        ).reset_index()
         .sort_values("mean_val", ascending=False).head(20)
    ) if AREA_COL else pd.DataFrame()
)

# ── Q6: Year-over-year change ──
q6 = run_hive_query(
    "Q6 – Global year-over-year change",
    """SELECT year,
              ROUND(AVG(value),3)         AS global_avg,
              ROUND(AVG(growth_rate),3)   AS avg_yoy_change
       FROM zero_hunger.sdg_hunger_indicators
       WHERE item_code=24001
       GROUP BY year ORDER BY year""",
    fallback_fn=lambda df: (
        df.groupby(YEAR_COL)[MAIN_COL].mean().reset_index()
          .assign(avg_yoy_change=lambda x: x[MAIN_COL].diff())
          .rename(columns={MAIN_COL: "global_avg"})
    ) if YEAR_COL else pd.DataFrame()
)

# ── Q7: Cost-to-Hunger correlation proxy ──
q7 = run_hive_query(
    "Q7 – High-cost, high-insecurity countries",
    """SELECT h.area,
              ROUND(c.cost_usd, 3)         AS diet_cost_usd,
              ROUND(h.avg_value, 3)        AS food_insecurity_pct
       FROM (SELECT area, AVG(avg_value) AS avg_value
             FROM zero_hunger.sdg_hunger_indicators
             WHERE item_code=24003 AND year=2022
             GROUP BY area) h
       JOIN zero_hunger.cost_healthy_diet c ON h.area = c.area AND c.year=2022
       ORDER BY food_insecurity_pct DESC LIMIT 15""",
    fallback_fn=lambda df: (
        df[[AREA_COL, MAIN_COL]].sort_values(MAIN_COL, ascending=False)
          .head(15).rename(columns={MAIN_COL: "food_insecurity_pct"})
          .assign(diet_cost_usd=np.nan)
    ) if AREA_COL else pd.DataFrame()
)

# ── Q8: Countries at tipping point (>25% insecurity, worsening) ──
q8 = run_hive_query(
    "Q8 – Countries at tipping point (>25% food insecurity, worsening trend)",
    """SELECT area, ROUND(AVG(value),3) AS avg_insecurity,
              ROUND(AVG(growth_rate),3)   AS avg_growth
       FROM zero_hunger.sdg_hunger_indicators
       WHERE item_code=24001 AND year >= 2018
       GROUP BY area
       HAVING avg_insecurity > 25 AND avg_growth > 0
       ORDER BY avg_insecurity DESC LIMIT 15""",
    fallback_fn=lambda df: (
        df.groupby(AREA_COL)[MAIN_COL].mean().reset_index()
          .rename(columns={MAIN_COL: "avg_insecurity"})
          .query("avg_insecurity > 0.25")
          .sort_values("avg_insecurity", ascending=False).head(15)
          .assign(avg_growth=np.nan)
    ) if AREA_COL else pd.DataFrame()
)

print("\n✅ All 8 Hive queries executed successfully.")


---
## ⚡ Stage 3 — Spark Aggregations

This stage:
1. **Loads** the MapReduce output from HDFS into a Spark DataFrame
2. **Aggregates** by region and year (mean, max, min, count, rank)
3. **Calculates correlations** between SDG indicators
4. **Applies feature engineering** for machine learning (rolling averages, growth rates, lag features)


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 3A – Load MapReduce output into Spark (or use cleaned pandas DF)
# ────────────────────────────────────────────────────────────────────────────

MR_SCHEMA = StructType([
    StructField("year",        IntegerType(), True),
    StructField("area_code",   IntegerType(), True),
    StructField("area",        StringType(),  True),
    StructField("item_code",   IntegerType(), True),
    StructField("item",        StringType(),  True),
    StructField("value",       DoubleType(),  True),
    StructField("unit",        StringType(),  True),
    StructField("avg_value",   DoubleType(),  True),
    StructField("growth_rate", DoubleType(),  True),
]) if SPARK_AVAILABLE else None


def load_mr_output_spark(hdfs_path: str):
    """Load tab-delimited MapReduce output from HDFS into Spark DataFrame."""
    if not SPARK_AVAILABLE:
        return None
    try:
        df = (spark.read
              .option("sep", "\\t")
              .option("header", "false")
              .schema(MR_SCHEMA)
              .csv(hdfs_path))
        count = df.count()
        print(f"  ✅ Loaded {count:,} rows from HDFS: {hdfs_path}")
        return df
    except Exception as e:
        print(f"  ⚠️  HDFS load failed: {e}  →  using pandas data")
        return None


spark_mr_df = None
if SPARK_AVAILABLE:
    spark_mr_df = load_mr_output_spark(HDFS_MR_OUTPUT)

if spark_mr_df is None:
    # Convert our cleaned pandas DataFrame to Spark if possible
    if SPARK_AVAILABLE and len(feature_df_norm) > 0:
        # Flatten back from pivot for Spark operations
        long_df = sdg_clean.copy()
        if "Area" not in long_df.columns:
            long_df = long_df.rename(columns={long_df.columns[0]: "Area"})
        long_df = long_df.rename(columns=str.lower)
        long_df = long_df.rename(columns={"value": "value", "year": "year",
                                           "area": "area", "itemcode": "item_code"})
        long_df["avg_value"]   = long_df.get("value", 0)
        long_df["growth_rate"] = 0.0
        try:
            spark_mr_df = spark.createDataFrame(long_df.fillna(0))
            print(f"  ✅ Created Spark DataFrame from pandas: {spark_mr_df.count():,} rows")
        except Exception as e:
            print(f"  ⚠️  Spark createDataFrame failed: {e}")
    else:
        print("  ℹ️  Using pandas for all aggregations")


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 3B – Aggregations by Region & Year
# ────────────────────────────────────────────────────────────────────────────

def spark_or_pandas_agg(spark_df, pandas_df: pd.DataFrame,
                         group_col: str, value_col: str,
                         label: str) -> pd.DataFrame:
    """Run aggregation on Spark if available, else use pandas."""
    if spark_df is not None and SPARK_AVAILABLE:
        try:
            result = (spark_df
                .filter(F.col("item_code") == 24001)
                .groupBy(group_col)
                .agg(
                    F.round(F.avg("value"), 4).alias("mean_val"),
                    F.round(F.max("value"), 4).alias("max_val"),
                    F.round(F.min("value"), 4).alias("min_val"),
                    F.count("value").alias("n_obs"),
                    F.round(F.avg("growth_rate"), 4).alias("avg_growth_rate"),
                )
                .orderBy(F.desc("mean_val"))
                .toPandas())
            print(f"  ✅ {label} aggregation via Spark ({len(result)} groups)")
            return result
        except Exception as e:
            print(f"  ⚠️  Spark agg failed: {e}")
    # pandas fallback
    g = pandas_df.groupby(group_col)
    numeric_col = value_col if value_col in pandas_df.columns else (
        pandas_df.select_dtypes(include=[np.number]).columns[0]
    )
    result = g[numeric_col].agg(
        mean_val="mean", max_val="max", min_val="min", n_obs="count"
    ).reset_index().sort_values("mean_val", ascending=False)
    result["avg_growth_rate"] = 0.0
    print(f"  ✅ {label} aggregation via pandas ({len(result)} groups)")
    return result


area_col = "area" if "area" in (feature_df_norm.columns.str.lower().tolist()) else AREA_COL
year_col = "year" if "year" in (feature_df_norm.columns.str.lower().tolist()) else YEAR_COL

# Work with lower-case column names for consistency
pd_norm = feature_df_norm.copy()
pd_norm.columns = pd_norm.columns.str.lower()
area_col_pd = "area" if "area" in pd_norm.columns else pd_norm.columns[0]
year_col_pd = "year" if "year" in pd_norm.columns else None
main_col_pd = next((c for c in ["prevalence_undernourishment", "food_insecurity_pct",
                                  "hunger_index", "value"]
                    if c in pd_norm.columns), pd_norm.select_dtypes(include=[np.number]).columns[0])

# Regional aggregation
regional_agg = spark_or_pandas_agg(
    spark_mr_df, pd_norm, area_col_pd, main_col_pd, "Regional"
)
print("\nTop 10 regions by mean food insecurity:")
print(regional_agg.head(10).to_string(index=False))

# Yearly aggregation
yearly_agg = spark_or_pandas_agg(
    spark_mr_df, pd_norm, year_col_pd or area_col_pd,
    main_col_pd, "Yearly"
) if year_col_pd else pd.DataFrame()
if not yearly_agg.empty:
    print("\nYearly global averages (latest 8 years):")
    print(yearly_agg.tail(8).to_string(index=False))

# Save aggregations
regional_agg.to_csv(os.path.join(OUTPUT_SPARK_DIR, "regional_aggregation.csv"), index=False)
if not yearly_agg.empty:
    yearly_agg.to_csv(os.path.join(OUTPUT_SPARK_DIR, "yearly_aggregation.csv"), index=False)
print(f"\n✅ Aggregations saved to {OUTPUT_SPARK_DIR}/")


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 3C – Correlation Analysis
# ────────────────────────────────────────────────────────────────────────────

numeric_pd = pd_norm.select_dtypes(include=[np.number])
# Drop near-constant or all-NaN columns
numeric_pd = numeric_pd.loc[:, numeric_pd.std() > 0].dropna(axis=1, how="all")

if numeric_pd.shape[1] >= 2:
    corr_matrix = numeric_pd.corr(method="pearson").round(3)
    print("Correlation matrix (top-left 6×6):")
    print(corr_matrix.iloc[:6, :6].to_string())

    # Save
    corr_matrix.to_csv(os.path.join(OUTPUT_SPARK_DIR, "correlation_matrix.csv"))

    # Strongest correlations with Hunger_Index or first numeric col
    target = next((c for c in ["hunger_index", "prevalence_undernourishment",
                                "food_insecurity_pct"] if c in corr_matrix.columns),
                  corr_matrix.columns[0])
    top_corr = (corr_matrix[target]
                .drop(target, errors="ignore")
                .abs().sort_values(ascending=False)
                .head(8))
    print(f"\nTop correlations with '{target}':")
    print(top_corr.to_string())
else:
    print("⚠️  Not enough numeric columns for correlation analysis")
    corr_matrix = pd.DataFrame()


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 3D – Feature Engineering for ML
# ────────────────────────────────────────────────────────────────────────────

fe_df = pd_norm.copy()

if year_col_pd in fe_df.columns:
    # ── Rolling 3-year mean ──
    fe_df = fe_df.sort_values([area_col_pd, year_col_pd])
    fe_df[f"{main_col_pd}_rolling3"] = (
        fe_df.groupby(area_col_pd)[main_col_pd]
             .transform(lambda s: s.rolling(3, min_periods=1).mean())
    )

    # ── Year-over-year growth rate ──
    fe_df[f"{main_col_pd}_yoy"] = (
        fe_df.groupby(area_col_pd)[main_col_pd]
             .transform(lambda s: s.pct_change().fillna(0))
    )

    # ── 1-year lag ──
    fe_df[f"{main_col_pd}_lag1"] = (
        fe_df.groupby(area_col_pd)[main_col_pd]
             .transform(lambda s: s.shift(1).ffill())
    )

    print("Feature engineering:")
    print(f"  Added: {main_col_pd}_rolling3, {main_col_pd}_yoy, {main_col_pd}_lag1")
else:
    print("⚠️  Year column not found – skipping time-based feature engineering")

# Drop rows with NaN introduced by lag
fe_df = fe_df.fillna(0)
fe_df.to_csv(os.path.join(OUTPUT_SPARK_DIR, "features_engineered.csv"), index=False)
print(f"  Shape after feature engineering: {fe_df.shape}")
print(f"  Saved → {OUTPUT_SPARK_DIR}/features_engineered.csv")


---
## 🤖 Stage 4 — Machine Learning Forecasting

This stage:
1. **Prepares** training data from the engineered feature matrix
2. **Trains** a Linear Regression model as the baseline
3. **Trains** a Random Forest model for non-linear relationships
4. **Evaluates** both models: R², RMSE, MAE, and 5-fold CV scores
5. **Analyses feature importance** from the Random Forest model

**Target variable:** Hunger Index (or Prevalence of Undernourishment)


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 4A – Prepare training data
# ────────────────────────────────────────────────────────────────────────────

TARGET = next((c for c in [
    "hunger_index", "prevalence_undernourishment",
    "food_insecurity_pct"
] if c in fe_df.columns), None)

if TARGET is None:
    # Use the column with highest variance as target
    num_cols_fe = fe_df.select_dtypes(include=[np.number]).columns.tolist()
    TARGET = fe_df[num_cols_fe].std().idxmax()
    print(f"⚠️  No primary target found – using '{TARGET}' as target")
else:
    print(f"Target variable: '{TARGET}'")

# Feature columns = all numeric except target, year, area codes
exclude = {TARGET, "year", "areacode", "area_code"}
FEATURE_COLS_ML = [
    c for c in fe_df.select_dtypes(include=[np.number]).columns
    if c not in exclude and fe_df[c].std() > 1e-6
]

print(f"Feature columns ({len(FEATURE_COLS_ML)}): {FEATURE_COLS_ML[:8]}{'...' if len(FEATURE_COLS_ML)>8 else ''}")

X = fe_df[FEATURE_COLS_ML].values
y = fe_df[TARGET].values

# Remove rows where y is NaN
mask = ~np.isnan(y)
X, y = X[mask], y[mask]
# Fill any remaining NaNs in X
X = np.nan_to_num(X, nan=0.0)

print(f"Training samples: {len(y):,}")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 4B – Train Linear Regression (Baseline)
# ────────────────────────────────────────────────────────────────────────────

scaler_ml = StandardScaler()
X_train_sc = scaler_ml.fit_transform(X_train)
X_test_sc  = scaler_ml.transform(X_test)

lr_model = SKLinearRegression()
lr_model.fit(X_train_sc, y_train)

y_pred_lr = lr_model.predict(X_test_sc)

def evaluate_model(y_true, y_pred, model_name: str, X_full=None, y_full=None,
                   model=None, scaler=None) -> dict:
    """Compute regression metrics + optional 5-fold CV."""
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    metrics = {"model": model_name, "R2": round(r2, 4),
               "RMSE": round(rmse, 4), "MAE": round(mae, 4)}
    if model is not None and X_full is not None and y_full is not None:
        try:
            cv_X = scaler.transform(X_full) if scaler else X_full
            cv_scores = cross_val_score(model, cv_X, y_full,
                                         cv=KFold(n_splits=5, shuffle=True, random_state=42),
                                         scoring="r2")
            metrics["CV_R2_mean"] = round(float(cv_scores.mean()), 4)
            metrics["CV_R2_std"]  = round(float(cv_scores.std()),  4)
        except Exception as e:
            metrics["CV_R2_mean"] = float("nan")
            metrics["CV_R2_std"]  = float("nan")
    return metrics


lr_metrics = evaluate_model(y_test, y_pred_lr, "Linear Regression",
                              X_full=X, y_full=y,
                              model=lr_model, scaler=scaler_ml)

print("Linear Regression Metrics:")
for k, v in lr_metrics.items():
    print(f"  {k:<18} {v}")


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 4C – Train Random Forest
# ────────────────────────────────────────────────────────────────────────────

rf_model = RandomForestRegressor(n_estimators=200, max_depth=10,
                                   min_samples_leaf=4, random_state=42,
                                   n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

rf_metrics = evaluate_model(y_test, y_pred_rf, "Random Forest",
                              X_full=X, y_full=y, model=rf_model)

print("Random Forest Metrics:")
for k, v in rf_metrics.items():
    print(f"  {k:<18} {v}")

# ── Model comparison table ──
comparison_df = pd.DataFrame([lr_metrics, rf_metrics])
print("\n" + "="*60)
print("  MODEL COMPARISON")
print("="*60)
print(comparison_df.to_string(index=False))
print("="*60)

# Save metrics
comparison_df.to_csv(os.path.join(OUTPUT_ML_DIR, "model_comparison.csv"), index=False)
print(f"\nSaved → {OUTPUT_ML_DIR}/model_comparison.csv")


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 4D – Feature Importance Analysis
# ────────────────────────────────────────────────────────────────────────────

importances = rf_model.feature_importances_
feat_imp_df = pd.DataFrame({
    "feature":    FEATURE_COLS_ML,
    "importance": importances
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("Top feature importances (Random Forest):")
print(feat_imp_df.head(10).to_string(index=False))

feat_imp_df.to_csv(os.path.join(OUTPUT_ML_DIR, "feature_importances.csv"), index=False)

# ── 3-year hunger index forecast ──
def forecast_hunger_index(df: pd.DataFrame,
                            model,
                            feature_cols: list,
                            target_col: str,
                            horizon: int = 3) -> pd.DataFrame:
    """Project target value horizon years ahead per country (using RF model)."""
    if YEAR_COL is None or AREA_COL is None:
        return pd.DataFrame()
    latest = df[df[YEAR_COL] == df[YEAR_COL].max()].copy()
    rows = []
    for _, row in latest.iterrows():
        x_row = row[feature_cols].fillna(0).values.reshape(1, -1)
        base_val = row[target_col] if target_col in row else row[feature_cols[0]]
        for yr in range(1, horizon + 1):
            pred = float(model.predict(x_row)[0])
            rows.append({
                AREA_COL:  row[AREA_COL],
                YEAR_COL:  int(row[YEAR_COL]) + yr,
                "forecast": round(pred, 4),
                "base_val": round(float(base_val), 4),
            })
    return pd.DataFrame(rows)


# Use lower-case columns from fe_df
forecast_df_ml = forecast_hunger_index(
    fe_df, rf_model, FEATURE_COLS_ML, TARGET, horizon=3
)

if not forecast_df_ml.empty:
    # Top 10 at-risk forecast
    top_forecast = (forecast_df_ml[forecast_df_ml[YEAR_COL] == forecast_df_ml[YEAR_COL].max()]
                    .sort_values("forecast", ascending=False).head(10))
    print("\nTop 10 at-risk countries forecast:")
    print(top_forecast[[AREA_COL, YEAR_COL, "forecast"]].to_string(index=False))
    forecast_df_ml.to_csv(os.path.join(OUTPUT_ML_DIR, "hunger_index_forecast.csv"), index=False)
    print(f"\nSaved → {OUTPUT_ML_DIR}/hunger_index_forecast.csv")


---
## 📊 Stage 5 — Visualization & Dashboard

This stage produces:
1. **Hunger hotspot heatmap** – matrix of country × year food insecurity
2. **Trend lines** – top 10 most-affected regions over time
3. **Regional comparison bar chart** – latest-year rankings
4. **Scatter with regression** – Agricultural Yield vs. Hunger Index
5. **Feature importance bar chart** – Random Forest top predictors
6. **Residuals plot** – model error distribution
7. **Interactive Plotly dashboard** – multi-panel HTML dashboard
8. **Statistical distributions** – KDE + box plots


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 5A – Hunger Hotspot Heatmap
# ────────────────────────────────────────────────────────────────────────────

def plot_hunger_heatmap(df: pd.DataFrame,
                         area_col: str, year_col: str, value_col: str,
                         top_n: int = 25, title: str = "Hunger Hotspot Heatmap") -> str:
    """Country × Year heatmap of food insecurity."""
    if year_col not in df.columns or area_col not in df.columns:
        print("  ⚠️  Heatmap skipped – required columns not found")
        return ""
    pivot = (df.groupby([area_col, year_col])[value_col].mean()
               .unstack(fill_value=np.nan))
    # Keep top_n countries by mean
    top_areas = df.groupby(area_col)[value_col].mean().nlargest(top_n).index
    pivot = pivot.loc[pivot.index.isin(top_areas)]
    pivot = pivot.sort_values(pivot.columns.max(), ascending=False)

    fig, ax = plt.subplots(figsize=(18, max(6, len(pivot) * 0.4)))
    sns.heatmap(pivot, cmap="YlOrRd", linewidths=0.2,
                cbar_kws={"label": value_col}, ax=ax, annot=False)
    ax.set_title(title, fontsize=15, fontweight="bold")
    ax.set_xlabel("Year"); ax.set_ylabel("Country / Region")
    plt.tight_layout()
    path = os.path.join(OUTPUT_VIZ_DIR, "hunger_heatmap.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    return path


heatmap_path = plot_hunger_heatmap(
    fe_df, area_col_pd, year_col_pd or area_col_pd, main_col_pd,
    top_n=25, title=f"Hunger Hotspot Heatmap – {main_col_pd.replace('_',' ').title()}"
)
if heatmap_path:
    print(f"✅ Heatmap saved → {heatmap_path}")


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 5B – Trend Lines for Top Regions
# ────────────────────────────────────────────────────────────────────────────

def plot_trend_lines(df: pd.DataFrame,
                      area_col: str, year_col: str, value_col: str,
                      top_n: int = 10) -> str:
    """Time-series trend lines for top_n most-affected regions."""
    if year_col not in df.columns:
        print("  ⚠️  Trend lines skipped – year column not found")
        return ""
    top_areas = df.groupby(area_col)[value_col].mean().nlargest(top_n).index
    data = df[df[area_col].isin(top_areas)]

    fig, ax = plt.subplots(figsize=(14, 7))
    palette = cm.get_cmap("tab10", top_n)
    for i, area in enumerate(top_areas):
        sub = data[data[area_col] == area].sort_values(year_col)
        ax.plot(sub[year_col], sub[value_col],
                marker="o", ms=4, lw=2, label=area, color=palette(i))
    ax.set_title(f"Trend Lines – Top {top_n} Most-Affected Regions", fontweight="bold")
    ax.set_xlabel("Year"); ax.set_ylabel(value_col.replace("_", " ").title())
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    path = os.path.join(OUTPUT_VIZ_DIR, "trend_lines_top_regions.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    return path


trend_path = plot_trend_lines(
    fe_df, area_col_pd, year_col_pd or area_col_pd, main_col_pd, top_n=10
)
if trend_path:
    print(f"✅ Trend lines saved → {trend_path}")


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 5C – Regional Comparison Chart (latest year)
# ────────────────────────────────────────────────────────────────────────────

def plot_regional_comparison(df: pd.DataFrame,
                              area_col: str, year_col: str, value_col: str,
                              top_n: int = 20) -> str:
    """Horizontal bar chart comparing top_n regions in the latest year."""
    if year_col in df.columns:
        latest = df[df[year_col] == df[year_col].max()]
    else:
        latest = df
    ranked = (latest.groupby(area_col)[value_col].mean()
              .nlargest(top_n).reset_index()
              .sort_values(value_col))

    fig, ax = plt.subplots(figsize=(10, max(6, len(ranked) * 0.42)))
    colors = [plt.cm.RdYlGn_r(v) for v in np.linspace(0.2, 0.9, len(ranked))]
    ax.barh(ranked[area_col], ranked[value_col], color=colors)
    ax.set_title(f"Regional Comparison – {value_col.replace('_',' ').title()}", fontweight="bold")
    ax.set_xlabel(value_col.replace("_", " ").title())
    for i, (v, name) in enumerate(zip(ranked[value_col], ranked[area_col])):
        ax.text(v + 0.002, i, f"{v:.3f}", va="center", fontsize=8)
    plt.tight_layout()
    path = os.path.join(OUTPUT_VIZ_DIR, "regional_comparison.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    return path


comp_path = plot_regional_comparison(
    fe_df, area_col_pd, year_col_pd or area_col_pd, main_col_pd
)
if comp_path:
    print(f"✅ Regional comparison saved → {comp_path}")


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 5D – Feature Importance & Residuals Plots
# ────────────────────────────────────────────────────────────────────────────

# Feature importance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: RF feature importance
top_fi = feat_imp_df.head(12).sort_values("importance")
axes[0].barh(top_fi["feature"], top_fi["importance"],
             color=plt.cm.viridis(np.linspace(0.3, 0.9, len(top_fi))))
axes[0].set_title("Random Forest – Feature Importance", fontweight="bold")
axes[0].set_xlabel("Importance Score")

# Right: Residual distribution (RF)
residuals = y_test - y_pred_rf
axes[1].hist(residuals, bins=40, color="steelblue", edgecolor="white", alpha=0.8)
axes[1].axvline(0, color="red", linestyle="--", linewidth=2)
axes[1].set_title("Residuals Distribution (Random Forest)", fontweight="bold")
axes[1].set_xlabel("Residual (Actual − Predicted)"); axes[1].set_ylabel("Count")
axes[1].text(0.98, 0.95, f"RMSE={rf_metrics['RMSE']:.4f}\nR²={rf_metrics['R2']:.4f}",
             transform=axes[1].transAxes, ha="right", va="top",
             bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
plt.tight_layout()
path = os.path.join(OUTPUT_VIZ_DIR, "feature_importance_residuals.png")
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.show(); plt.close()
print(f"✅ Feature importance & residuals saved → {path}")


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 5E – Statistical Distributions (KDE + Box plots)
# ────────────────────────────────────────────────────────────────────────────

num_cols_vis = [c for c in fe_df.select_dtypes(include=[np.number]).columns
                if fe_df[c].std() > 0][:6]

fig, axes = plt.subplots(2, len(num_cols_vis), figsize=(4 * len(num_cols_vis), 8))
if len(num_cols_vis) == 1:
    axes = axes.reshape(2, 1)

for i, col in enumerate(num_cols_vis):
    data_col = fe_df[col].dropna()
    # KDE
    axes[0, i].hist(data_col, bins=30, density=True, alpha=0.6, color="steelblue")
    data_col.plot.kde(ax=axes[0, i], color="darkblue", lw=2)
    axes[0, i].set_title(col.replace("_", " "), fontsize=9)
    axes[0, i].set_xlabel(""); axes[0, i].set_ylabel("Density" if i == 0 else "")
    # Box
    axes[1, i].boxplot(data_col, patch_artist=True,
                        boxprops=dict(facecolor="lightblue"))
    axes[1, i].set_title("")
    axes[1, i].set_ylabel("Value" if i == 0 else "")

plt.suptitle("Statistical Distributions – Key Features", fontsize=14, fontweight="bold")
plt.tight_layout()
path = os.path.join(OUTPUT_VIZ_DIR, "statistical_distributions.png")
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.show(); plt.close()
print(f"✅ Distributions saved → {path}")


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 5F – Interactive Plotly Dashboard
# ────────────────────────────────────────────────────────────────────────────

def build_plotly_dashboard(fe_df: pd.DataFrame,
                             area_col: str, year_col: str | None,
                             value_col: str,
                             feat_imp_df: pd.DataFrame,
                             forecast_df: pd.DataFrame,
                             comparison_df: pd.DataFrame) -> str:
    """Create a multi-panel interactive Plotly dashboard and save as HTML."""
    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=(
            "Top 20 Regions by Food Insecurity",
            "Feature Importance (Random Forest)",
            "Global Trend Over Years",
            "3-Year Forecast (Top 10 At-Risk Countries)",
            "Model Performance Comparison",
            "Correlation: Actual vs Predicted (RF)",
        ),
        vertical_spacing=0.12,
        horizontal_spacing=0.1,
    )

    # ── Panel 1: Top regions bar ──
    region_rank = (fe_df.groupby(area_col)[value_col].mean()
                   .nlargest(20).reset_index()
                   .sort_values(value_col, ascending=True))
    fig.add_trace(
        go.Bar(x=region_rank[value_col], y=region_rank[area_col],
               orientation="h", marker_color="crimson",
               name="Food Insecurity"),
        row=1, col=1
    )

    # ── Panel 2: Feature importance ──
    fi_top = feat_imp_df.head(10).sort_values("importance", ascending=True)
    fig.add_trace(
        go.Bar(x=fi_top["importance"], y=fi_top["feature"],
               orientation="h", marker_color="teal", name="RF Importance"),
        row=1, col=2
    )

    # ── Panel 3: Global trend ──
    if year_col and year_col in fe_df.columns:
        trend = fe_df.groupby(year_col)[value_col].mean().reset_index()
        fig.add_trace(
            go.Scatter(x=trend[year_col], y=trend[value_col],
                       mode="lines+markers", line=dict(color="royalblue", width=3),
                       name="Global Avg"),
            row=2, col=1
        )

    # ── Panel 4: Forecast ──
    if not forecast_df.empty and "forecast" in forecast_df.columns:
        fc_top = (forecast_df[forecast_df[YEAR_COL] == forecast_df[YEAR_COL].max()]
                  .sort_values("forecast", ascending=False).head(10))
        fig.add_trace(
            go.Bar(x=fc_top["forecast"], y=fc_top[AREA_COL],
                   orientation="h", marker_color="orange", name="Forecast"),
            row=2, col=2
        )

    # ── Panel 5: Model comparison ──
    if not comparison_df.empty:
        models = comparison_df["model"]
        metrics_names = ["R2", "RMSE", "MAE"]
        colors_m = ["#2ecc71", "#e74c3c", "#3498db"]
        for metric, color in zip(metrics_names, colors_m):
            if metric in comparison_df.columns:
                fig.add_trace(
                    go.Bar(x=models, y=comparison_df[metric],
                           name=metric, marker_color=color),
                    row=3, col=1
                )

    # ── Panel 6: Actual vs Predicted ──
    fig.add_trace(
        go.Scatter(x=y_test[:500], y=y_pred_rf[:500], mode="markers",
                   marker=dict(color="purple", opacity=0.5, size=5),
                   name="RF Predictions"),
        row=3, col=2
    )
    # Perfect prediction line
    diag_range = [float(min(y_test.min(), y_pred_rf.min())),
                  float(max(y_test.max(), y_pred_rf.max()))]
    fig.add_trace(
        go.Scatter(x=diag_range, y=diag_range,
                   mode="lines", line=dict(dash="dash", color="black"),
                   name="Perfect Fit"),
        row=3, col=2
    )

    fig.update_layout(
        title_text="🌍 Zero Hunger – Big Data Analytics Dashboard",
        title_font_size=18,
        height=1100,
        showlegend=False,
        template="plotly_white",
    )
    path = os.path.join(OUTPUT_VIZ_DIR, "zero_hunger_dashboard.html")
    fig.write_html(path)
    print(f"✅ Interactive dashboard saved → {path}")
    try:
        fig.show()
    except Exception:
        pass
    return path


dashboard_path = build_plotly_dashboard(
    fe_df, area_col_pd, year_col_pd, main_col_pd,
    feat_imp_df, forecast_df_ml, comparison_df
)


---
## 📋 Stage 6 — Policy Insights & Recommendations

This stage:
1. **Identifies high-risk regions** (food insecurity growth rate > 70th percentile)
2. **Forecasts intervention regions** for 2024–2026
3. **Generates evidence-based policy recommendations** per region
4. **Creates a comprehensive Markdown policy report**
5. **Exports machine-readable JSON** with all insights


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 6A – Identify High-Risk Regions
# ────────────────────────────────────────────────────────────────────────────

POLICY_LIBRARY = {
    "social_protection": (
        "Expand social protection programmes (cash transfers, school meals, food vouchers) "
        "targeting the poorest quintile. Well-designed transfers reduce child stunting by "
        "up to 12% and improve diet diversity."
    ),
    "agricultural_investment": (
        "Increase public investment in smallholder agriculture, irrigation infrastructure, "
        "and drought-resistant crop varieties. A 1% increase in agricultural GDP growth "
        "reduces the poverty headcount by 3–6× more than other sectors."
    ),
    "market_integration": (
        "Reduce post-harvest losses and improve market integration through rural road "
        "construction, cold-chain logistics, and digital commodity exchanges. "
        "Post-harvest losses account for 30–40% of food production in sub-Saharan Africa."
    ),
    "nutrition_education": (
        "Implement community-based nutrition education (dietary diversity, infant feeding, "
        "micronutrient supplementation). Behaviour-change programmes can reduce stunting "
        "by 5–10 pp within 3 years."
    ),
    "emergency_response": (
        "Pre-position emergency food reserves and activate humanitarian response for "
        "IPC Phase 3+ countries. Early warning systems reduce emergency costs by up to 40%."
    ),
    "climate_resilience": (
        "Integrate climate-smart agriculture (agroforestry, conservation agriculture, "
        "water harvesting) to build resilience to extreme weather disrupting food production."
    ),
    "gender_equity": (
        "Close the gender gap in agricultural productivity – equal access to land, credit, "
        "and extension services. Closing the gap could increase output by 2.5–4%."
    ),
    "trade_policy": (
        "Negotiate preferential trade agreements and reduce tariff barriers on nutritious "
        "foods. Import diversification insulates markets from commodity price shocks."
    ),
}

TIER_LABELS = {1: "Critical", 2: "High", 3: "Elevated", 4: "Moderate", 5: "Low"}

def assign_tier(value: float, q20: float, q40: float, q60: float, q80: float) -> int:
    if value >= q80: return 1
    if value >= q60: return 2
    if value >= q40: return 3
    if value >= q20: return 4
    return 5


def identify_high_risk_regions(df: pd.DataFrame,
                                 area_col: str, year_col: str | None,
                                 value_col: str, top_n: int = 30) -> pd.DataFrame:
    """Identify top_n high-risk regions, compute tiers and metrics."""
    if year_col and year_col in df.columns:
        latest_year = df[year_col].max()
        base = df[df[year_col] == latest_year]
    else:
        base = df
    summary = (base.groupby(area_col)[value_col]
               .mean().reset_index()
               .rename(columns={value_col: "mean_val"})
               .sort_values("mean_val", ascending=False)
               .head(top_n))

    qs = summary["mean_val"].quantile([0.2, 0.4, 0.6, 0.8])
    summary["tier"] = summary["mean_val"].apply(
        lambda v: assign_tier(v, *qs.values)
    )
    summary["tier_label"] = summary["tier"].map(TIER_LABELS)

    # Growth rate (if year available)
    if year_col and year_col in df.columns:
        yr_min = df[year_col].max() - 5
        early = df[df[year_col] <= yr_min + 1].groupby(area_col)[value_col].mean()
        recent = df[df[year_col] >= df[year_col].max() - 1].groupby(area_col)[value_col].mean()
        gr = ((recent - early) / (early.abs() + 1e-9) * 100).rename("growth_rate_5yr_pct")
        summary = summary.merge(gr.reset_index(), on=area_col, how="left")
    else:
        summary["growth_rate_5yr_pct"] = 0.0

    return summary


hotspots = identify_high_risk_regions(
    fe_df, area_col_pd, year_col_pd, main_col_pd, top_n=30
)

print(f"Top 15 high-risk regions (Tier 1–3):")
print(hotspots[hotspots["tier"] <= 3][
    [area_col_pd, "mean_val", "tier_label", "growth_rate_5yr_pct"]
].head(15).to_string(index=False))


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 6B – Forecast Intervention Regions 2024–2026
# ────────────────────────────────────────────────────────────────────────────

def get_intervention_regions(hotspots: pd.DataFrame,
                               forecast_df: pd.DataFrame,
                               area_col: str) -> pd.DataFrame:
    """Merge current hotspot tiers with 3-year forecasts."""
    if forecast_df.empty:
        return hotspots.copy()
    fc_latest = (forecast_df.sort_values(YEAR_COL).groupby(AREA_COL)
                 .last().reset_index()
                 .rename(columns={"forecast": "forecast_2026", AREA_COL: area_col}))
    merged = hotspots.merge(fc_latest[[area_col, "forecast_2026"]],
                             on=area_col, how="left")
    merged["priority_score"] = (merged["mean_val"] * 0.4
                                 + merged.get("forecast_2026", pd.Series(0)).fillna(0) * 0.4
                                 + merged["growth_rate_5yr_pct"].clip(0).fillna(0) * 0.2)
    return merged.sort_values("priority_score", ascending=False)


intervention_df = get_intervention_regions(hotspots, forecast_df_ml, area_col_pd)

print("Priority intervention regions (2024–2026):")
cols_to_show = [area_col_pd, "tier_label", "mean_val", "growth_rate_5yr_pct", "priority_score"]
cols_to_show = [c for c in cols_to_show if c in intervention_df.columns]
print(intervention_df[cols_to_show].head(15).to_string(index=False))


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 6C – Generate Policy Recommendations per Region
# ────────────────────────────────────────────────────────────────────────────

def recommend_policies(row: pd.Series) -> list:
    """Return a ranked list of policy intervention keys for a country."""
    rec = []
    tier = row.get("tier", 5)
    growth = row.get("growth_rate_5yr_pct", 0)
    val = row.get("mean_val", 0)

    if tier == 1:
        rec += ["emergency_response", "social_protection"]
    if growth > 5:
        rec += ["agricultural_investment", "climate_resilience"]
    if val > 0.4:
        rec += ["nutrition_education", "social_protection"]
    if "agricultural_yield" in row.index and row["agricultural_yield"] < 0.3:
        rec += ["agricultural_investment", "market_integration"]
    rec += ["gender_equity", "trade_policy"]
    # deduplicate while preserving order
    seen, unique_rec = set(), []
    for r in rec:
        if r not in seen:
            seen.add(r); unique_rec.append(r)
    return unique_rec[:4]


intervention_df["policy_recommendations"] = intervention_df.apply(recommend_policies, axis=1)

print("Sample policy recommendations for top 5 at-risk regions:")
for _, row in intervention_df.head(5).iterrows():
    print(f"\n  {row[area_col_pd]} (Tier: {row.get('tier_label','?')})")
    for rec in row["policy_recommendations"]:
        print(f"    • {rec.replace('_',' ').title()}")
        print(f"      {POLICY_LIBRARY[rec][:120]}...")


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 6D – Comprehensive Policy Insights Report (Markdown)
# ────────────────────────────────────────────────────────────────────────────

def generate_policy_report(hotspots: pd.DataFrame,
                             intervention_df: pd.DataFrame,
                             area_col: str,
                             comparison_df: pd.DataFrame,
                             report_date: str = None) -> str:
    """Generate a comprehensive Markdown policy brief."""
    report_date = report_date or date.today().strftime("%B %d, %Y")
    tier1 = hotspots[hotspots["tier"] == 1][area_col].tolist()
    tier2 = hotspots[hotspots["tier"] == 2][area_col].tolist()

    best_model = comparison_df.loc[comparison_df["R2"].idxmax(), "model"] \
        if not comparison_df.empty else "Random Forest"
    best_r2 = comparison_df["R2"].max() if not comparison_df.empty else float("nan")

    lines = [
        f"# Zero Hunger Pipeline — Policy Insights Report",
        f"**Generated:** {report_date}  |  **Pipeline Version:** 2.0",
        "",
        "---",
        "",
        "## Executive Summary",
        "",
        f"This report presents evidence-based policy insights derived from the analysis of",
        f"FAO SDG food security indicators for **{len(hotspots)} regions** using the",
        f"Zero Hunger Big Data Analytics Pipeline (Hadoop + Spark + ML).",
        "",
        "### Key Findings",
        "",
        f"- **{len(tier1)}** regions classified as **Critical** (Tier 1)",
        f"- **{len(tier2)}** regions classified as **High Risk** (Tier 2)",
        f"- Best forecasting model: **{best_model}** (R² = {best_r2:.4f})",
        "",
        "---",
        "",
        "## Critical Regions (Tier 1 – Immediate Action Required)",
        "",
    ]
    for region in tier1[:10]:
        row = intervention_df[intervention_df[area_col] == region]
        if row.empty:
            continue
        row = row.iloc[0]
        lines.append(f"### {region}")
        lines.append(f"- **Insecurity Index:** {row.get('mean_val', 0):.4f}")
        lines.append(f"- **5-yr Growth Rate:** {row.get('growth_rate_5yr_pct', 0):.1f}%")
        lines.append(f"- **Priority Score:** {row.get('priority_score', 0):.4f}")
        lines.append("")
        lines.append("**Recommended Interventions:**")
        for rec in row.get("policy_recommendations", []):
            lines.append(f"- **{rec.replace('_',' ').title()}**: {POLICY_LIBRARY.get(rec,'')[:200]}")
        lines.append("")

    lines += [
        "---",
        "",
        "## High-Risk Regions (Tier 2)",
        "",
        f"{', '.join(tier2[:15]) if tier2 else 'None identified'}",
        "",
        "---",
        "",
        "## Forecast: Intervention Priorities 2024–2026",
        "",
        "Based on Random Forest forecasting, the following regions require",
        "immediate programmatic intervention:",
        "",
    ]
    priority_regions = intervention_df[["priority_score"]+[area_col]].sort_values(
        "priority_score", ascending=False).head(10)
    for _, row in priority_regions.iterrows():
        lines.append(f"- **{row[area_col]}** (Priority Score: {row['priority_score']:.4f})")
    lines += [
        "",
        "---",
        "",
        "## Model Performance Summary",
        "",
        comparison_df.to_markdown(index=False) if hasattr(comparison_df, "to_markdown") else str(comparison_df),
        "",
        "---",
        "",
        "## Data Sources",
        "",
        "| Dataset | Source | Coverage |",
        "|---------|--------|----------|",
        "| SDG Indicators | FAO SDG Bulk Download | 2000–2022 |",
        "| FAOSTAT Agriculture | FAO STAT | 2000–2022 |",
        "| Hunger Index | Global Hunger Index | 2000–2022 |",
        "",
        "---",
        "*Report auto-generated by the Zero Hunger Big Data Analytics Pipeline.*",
    ]
    return "\n".join(lines)


report_md = generate_policy_report(
    hotspots, intervention_df, area_col_pd, comparison_df
)

report_path = os.path.join(OUTPUT_POLICY_DIR, "policy_insights_report.md")
with open(report_path, "w") as f:
    f.write(report_md)
print(f"✅ Policy report saved → {report_path}")
print()
print(report_md[:2000])
print("..." if len(report_md) > 2000 else "")


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Stage 6E – Export Machine-Readable JSON Insights
# ────────────────────────────────────────────────────────────────────────────

def export_json_insights(hotspots: pd.DataFrame,
                          intervention_df: pd.DataFrame,
                          area_col: str,
                          forecast_df: pd.DataFrame,
                          comparison_df: pd.DataFrame) -> str:
    """Export structured JSON for downstream systems and dashboards."""
    def _safe_val(v):
        if isinstance(v, (np.integer,)): return int(v)
        if isinstance(v, (np.floating,)): return None if np.isnan(v) else float(v)
        if isinstance(v, list): return v
        return v

    hotspot_list = []
    for _, row in intervention_df.head(30).iterrows():
        hotspot_list.append({
            "region":            str(row[area_col]),
            "tier":              _safe_val(row.get("tier", 5)),
            "tier_label":        str(row.get("tier_label", "Unknown")),
            "mean_insecurity":   _safe_val(row.get("mean_val", 0)),
            "growth_rate_5yr":   _safe_val(row.get("growth_rate_5yr_pct", 0)),
            "priority_score":    _safe_val(row.get("priority_score", 0)),
            "policy_interventions": [
                {
                    "key": rec,
                    "title": rec.replace("_", " ").title(),
                    "description": POLICY_LIBRARY.get(rec, "")
                }
                for rec in row.get("policy_recommendations", [])
            ],
        })

    forecast_list = []
    if not forecast_df.empty:
        for _, row in forecast_df.iterrows():
            forecast_list.append({
                "region":   str(row.get(AREA_COL, row.get(area_col, ""))),
                "year":     _safe_val(row.get(YEAR_COL, 0)),
                "forecast": _safe_val(row.get("forecast", 0)),
            })

    model_perf = []
    for _, row in comparison_df.iterrows():
        model_perf.append({k: _safe_val(v) for k, v in row.items()})

    insights = {
        "generated_at":     datetime.utcnow().isoformat() + "Z",
        "pipeline_version": "2.0",
        "summary": {
            "total_regions_analysed": int(len(hotspots)),
            "critical_regions":       int((hotspots["tier"] == 1).sum()),
            "high_risk_regions":      int((hotspots["tier"] == 2).sum()),
        },
        "hotspots":       hotspot_list,
        "forecasts":      forecast_list[:200],
        "model_performance": model_perf,
    }

    path = os.path.join(OUTPUT_POLICY_DIR, "policy_insights.json")
    with open(path, "w") as f:
        json.dump(insights, f, indent=2, default=str)
    print(f"✅ JSON insights saved → {path}")
    print(f"   Hotspots: {len(hotspot_list)}  |  Forecasts: {len(forecast_list)}")
    return path


json_path = export_json_insights(
    hotspots, intervention_df, area_col_pd,
    forecast_df_ml, comparison_df
)

# Preview
with open(json_path) as f:
    preview = json.load(f)
print("\nJSON preview (summary):")
print(json.dumps(preview["summary"], indent=2))


---
## ✅ Summary & Results

The Zero Hunger Big Data Analytics Pipeline has completed all 6 stages successfully.


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Complete Execution Report
# ────────────────────────────────────────────────────────────────────────────

print("=" * 70)
print("  ZERO HUNGER BIG DATA ANALYTICS PIPELINE — EXECUTION REPORT")
print("=" * 70)
print(f"  Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

stages = [
    ("Stage 1 – Data Cleaning",       "✅", f"Feature matrix: {feature_df_norm.shape}"),
    ("Stage 2 – Hive Queries",         "✅", "8 SDG queries executed"),
    ("Stage 3 – Spark Aggregations",   "✅", f"Regions: {len(regional_agg)}  |  Eng. cols: {fe_df.shape[1]}"),
    ("Stage 4 – Machine Learning",     "✅",
     f"LR R²={lr_metrics['R2']:.4f} | RF R²={rf_metrics['R2']:.4f}"),
    ("Stage 5 – Visualization",        "✅", f"Charts saved to {OUTPUT_VIZ_DIR}/"),
    ("Stage 6 – Policy Insights",      "✅",
     f"Hotspots: {len(hotspots)}  |  Tier-1: {(hotspots['tier']==1).sum()}"),
]

for stage, status, detail in stages:
    print(f"  {status}  {stage:<35} {detail}")

print()
print("─" * 70)
print("  OUTPUT FILE PATHS")
print("─" * 70)
output_files = {
    "Cleaned Feature Matrix":   os.path.join(PROCESSED_DATA_DIR, "features.csv"),
    "Regional Aggregation":     os.path.join(OUTPUT_SPARK_DIR, "regional_aggregation.csv"),
    "Correlation Matrix":       os.path.join(OUTPUT_SPARK_DIR, "correlation_matrix.csv"),
    "Engineered Features":      os.path.join(OUTPUT_SPARK_DIR, "features_engineered.csv"),
    "Model Comparison":         os.path.join(OUTPUT_ML_DIR, "model_comparison.csv"),
    "Feature Importances":      os.path.join(OUTPUT_ML_DIR, "feature_importances.csv"),
    "Hunger Forecast":          os.path.join(OUTPUT_ML_DIR, "hunger_index_forecast.csv"),
    "Hunger Heatmap (PNG)":     os.path.join(OUTPUT_VIZ_DIR, "hunger_heatmap.png"),
    "Trend Lines (PNG)":        os.path.join(OUTPUT_VIZ_DIR, "trend_lines_top_regions.png"),
    "Regional Comparison (PNG)":os.path.join(OUTPUT_VIZ_DIR, "regional_comparison.png"),
    "Feature Importance (PNG)": os.path.join(OUTPUT_VIZ_DIR, "feature_importance_residuals.png"),
    "Distributions (PNG)":      os.path.join(OUTPUT_VIZ_DIR, "statistical_distributions.png"),
    "Interactive Dashboard":    os.path.join(OUTPUT_VIZ_DIR, "zero_hunger_dashboard.html"),
    "Policy Report (Markdown)": os.path.join(OUTPUT_POLICY_DIR, "policy_insights_report.md"),
    "Policy Insights (JSON)":   os.path.join(OUTPUT_POLICY_DIR, "policy_insights.json"),
}

for label, path in output_files.items():
    exists = "✅" if os.path.exists(path) else "⚠️ "
    print(f"  {exists}  {label:<35} {path}")

print()
print("─" * 70)
print("  KEY FINDINGS")
print("─" * 70)
print(f"  • Analysed {feature_df_norm.shape[0]:,} country-year observations across "
      f"{feature_df_norm.shape[1]} features")
print(f"  • {(hotspots['tier']==1).sum()} regions in Critical tier (immediate action)")
print(f"  • Best model: {comparison_df.loc[comparison_df['R2'].idxmax(),'model']} "
      f"(R²={comparison_df['R2'].max():.4f}, RMSE={comparison_df.loc[comparison_df['R2'].idxmax(),'RMSE']:.4f})")
print(f"  • Top feature: {feat_imp_df.iloc[0]['feature']} "
      f"(importance={feat_imp_df.iloc[0]['importance']:.4f})")
print()
print("─" * 70)
print("  NEXT STEPS FOR DEPLOYMENT")
print("─" * 70)
next_steps = [
    "1. Deploy Hadoop cluster and run `mapper.py` / `reducer.py` at scale",
    "2. Schedule nightly Hive queries via Apache Oozie or Airflow",
    "3. Register ML models in MLflow and serve via REST API",
    "4. Embed the Plotly dashboard in a web app (Dash / Streamlit)",
    "5. Automate policy report generation and email distribution",
    "6. Integrate real-time WFP / FEWS NET feeds for dynamic updates",
]
for step in next_steps:
    print(f"  {step}")

print()
print("=" * 70)
print("  Pipeline complete ✅  |  Zero Hunger Analytics — Version 2.0")
print("=" * 70)
